In [0]:
bronze_path = "/Volumes/workspace/default/medallion/bronze/orders"
silver_path = "/Volumes/workspace/default/medallion/silver/orders"
gold_path   = "/Volumes/workspace/default/medallion/gold/orders"

In [0]:
from pyspark.sql import SparkSession
t
spark = SparkSession.builder.appName("FlipkartPipeline").getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id","customer_id","product","category","city","date","amount","quantity"]

df = spark.createDataFrame(data, columns)

In [0]:
df.write.format("delta").mode("append").save(bronze_path)

In [0]:
from pyspark.sql.functions import col, to_date
df= spark.read.format("delta").load(bronze_path)

df = df.withColumn("amount", col("amount").cast("int")) \
       .withColumn("date", to_date(col("date")))

In [0]:
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


4. SILVER LAYER
Data Cleaning, 
Handle NULL values (amount, city), 
Decide: fill or remove


In [0]:
from pyspark.sql.functions import *

df = df.fillna({"amount": 0, "city": "Unknown"})
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df = df.filter(col("amount")>=0)
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df = df.dropDuplicates()
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
from pyspark.sql.window import Window
windowspec = Window.partitionBy("order_id").orderBy(col("date").desc())

silver_df = df.withColumn("rn", row_number().over(windowspec)).filter(col("rn")==1).drop("rn")
silver_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
silver_df.write.format("delta").mode("overwrite").save(silver_path)

In [0]:
silver_df = spark.read.format("delta").load(silver_path)
silver_df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


5. GOLD LAYER 
Create business-level insights
Tasks:
Aggregations: 
Total sales by product,
Total sales by category, 
Total sales by city 

In [0]:
from pyspark.sql.functions import *
product_sales = silver_df.groupBy("product").agg(sum(col("amount")).alias("total_sales"))
product_sales.display()

product,total_sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,105000
Headphones,3000


In [0]:
category_sales = silver_df.groupBy("category").agg(sum(col("amount")).alias("total_sales"))
category_sales.display()

category,total_sales
Electronics,160000
Accessories,11000


In [0]:
city_sales = silver_df.groupBy("city").agg(sum(col("amount")).alias("total_sales"))
city_sales.display()


city,total_sales
Delhi,55000
Chennai,33000
Unknown,3000
Hyderabad,72000
Mumbai,8000


Customer Metrics
Total orders per customer
Total spending per customer

In [0]:
customer_orders = silver_df.groupBy("customer_id").agg(count(col("order_id")).alias("total_orders"))
customer_orders.display()                                                  

customer_id,total_orders
C005,1
C004,1
C003,1
C001,2
C002,2
C007,1


In [0]:
customer_spends = silver_df.groupBy("customer_id").agg(sum(col("amount")).alias("total_spend"))
customer_spends.display()

customer_id,total_spend
C005,3000
C004,8000
C003,55000
C001,72000
C002,18000
C007,15000


Top selling product

In [0]:
top_product = product_sales.orderBy(col("total_sales").desc()).limit(1)
top_product.display()

product,total_sales
Laptop,105000


Top customer

In [0]:
top_customer = customer_spends.orderBy(col("total_spend").desc()).limit(1)
top_customer.display()

customer_id,total_spend
C001,72000


In [0]:
product_sales.write.format("delta").mode("overwrite").save(gold_path + "/product_sales")
category_sales.write.format("delta").mode("overwrite").save(gold_path + "/category_sales")
city_sales.write.format("delta").mode("overwrite").save(gold_path + "/city_sales")
customer_spends.write.format("delta").mode("overwrite").save(gold_path + "/customer_spend")
customer_orders.write.format("delta").mode("overwrite").save(gold_path + "/customer_orders")
top_product.write.format("delta").mode("overwrite").save(gold_path + "/top_product")
top_customer.write.format("delta").mode("overwrite").save(gold_path + "/top_customer")